# 🦜 LangChain v1: Your First Agent

## Learning Objectives
In this notebook, you will learn:
1. **Verifying your install** - confirm you are actually on LangChain 1.x, since the agent API differs completely from 0.x
2. **Loading credentials** - read `OPENAI_API_KEY` from a `.env` file with `python-dotenv`
3. **`create_agent`** - the single entry point for agents in v1, replacing `AgentExecutor` and `initialize_agent`
4. **Invoking an agent** - the `{"messages": [...]}` input shape and the string shorthand
5. **Reading the message trail** - why the agent returns four messages and one of them looks empty

## Prerequisites
- Python >= 3.11
- `pip install langchain langchain-openai python-dotenv`
- A `.env` file at the project root containing `OPENAI_API_KEY=sk-...`

---
## 🔍 Part 1: Verify the LangChain Version

Everything in this notebook is LangChain **1.x** API. On 0.x, `langchain.agents.create_agent`
does not exist at all, so check the version first — a wrong version here produces a confusing
`ImportError` three cells later rather than an obvious one.

In [ ]:
# ============================================================================
# VERSION CHECK: Confirm we are on LangChain 1.x
# ============================================================================
import langchain

print(f"✅ LangChain version: {langchain.__version__}")
assert langchain.__version__.startswith("1."), "This notebook requires LangChain 1.x"

---
## 🔑 Part 2: Environment Setup

Credentials live in a `.env` file, never in the notebook. `load_dotenv()` reads that file
and populates `os.environ`, so any LangChain provider package picks the key up automatically.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API credentials from .env
# ============================================================================
import os

from dotenv import load_dotenv

load_dotenv()  # reads .env from the current working directory (and parents)

assert os.getenv("OPENAI_API_KEY"), "❌ Set OPENAI_API_KEY in your .env file"
print("✅ Environment loaded successfully!")

---
## 🤖 Part 3: Build an Agent with `create_agent`

`create_agent` is the **one** way to build an agent in LangChain 1.0. It replaces the whole
0.x zoo — `AgentExecutor`, `initialize_agent`, `create_react_agent`, and
`langgraph.prebuilt.create_react_agent` — with a single function built directly on the
LangGraph runtime.

### Key Concepts:
- **`model`**: a model identifier string (`"gpt-5"`, or `"openai:gpt-4.1"` with an explicit
  provider prefix) or an already-constructed chat model object
- **`tools`**: a plain list of Python functions. A bare function works — the docstring becomes
  the tool description the model sees, and the type hints become its argument schema
- **`system_prompt`**: your instructions, now an explicit argument instead of the hidden
  template scaffolding you had to reverse-engineer in 0.x

> **Note**: `create_agent` returns a compiled LangGraph, so it supports `.invoke()`,
> `.stream()`, and `.get_graph()` like any other graph.

In [ ]:
# ============================================================================
# AGENT DEFINITION: A single-tool weather agent
# ============================================================================
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get the weather for a city."""
    # Stubbed for the demo — swap in a real weather API call.
    return f"The weather in {city} is sunny."


agent = create_agent(
    model="gpt-5",                                  # try "openai:gpt-4.1" to be explicit
    tools=[get_weather],                            # plain functions are valid tools
    system_prompt="You are a helpful assistant.",   # explicit, not hidden scaffolding
)

print("🤖 Agent built successfully!")
print(f"📋 Graph nodes: {list(agent.get_graph().nodes)}")

---
## ▶️ Part 4: Invoke the Agent

An agent takes a state dict whose `messages` key holds the conversation. The canonical form
is a list of role/content dicts:

```python
{"messages": [{"role": "user", "content": "..."}]}
```

In [ ]:
# ============================================================================
# RUN THE AGENT: Canonical message-list input
# ============================================================================
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather like in New York?"}]}
)

print("🤖 Final answer:", response["messages"][-1].content)

### 4.1 📋 Inspecting the Full Message Trail

`response["messages"]` holds every message from the run, not just the answer. A single tool
call produces **four** messages.

In [ ]:
# ============================================================================
# MESSAGE TRAIL: Every message the agent produced
# ============================================================================
response["messages"]

### 4.2 ❓ Why Does One AI Message Look Empty?

Printed as plain text, the trail looks broken — the first AI turn is blank:

```
[human] What is the weather like in New York?
[ai]
[tool] The weather in New York is sunny.
[ai] The weather in New York is sunny.
```

That is not a bug: **that AI message *is* the tool call.**

| # | Message | What it carries |
|---|---------|-----------------|
| 1 | `HumanMessage` | your question |
| 2 | `AIMessage` | `content=""`, plus `tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'call_...'}]` |
| 3 | `ToolMessage` | `"The weather in New York is sunny."` — what the function returned |
| 4 | `AIMessage` | the final natural-language answer |

When the model decides to call a tool it emits *only* the call; there is no prose to say
alongside it, so `.content` is an empty string and the real payload lives in `.tool_calls`.
Print `.content` alone and you see a blank line.

> **Note**: some models (Anthropic, and OpenAI's reasoning models) *do* emit a short
> "Let me check the weather…" preamble next to the call. Never assume an `AIMessage` is
> either text or a tool call — it can be both, or neither.

In [ ]:
# ============================================================================
# MESSAGE ANATOMY: Show content AND tool_calls side by side
# ============================================================================
for i, m in enumerate(response["messages"]):
    print(f"[{i}] {type(m).__name__:<12} content={m.content!r}")
    if getattr(m, "tool_calls", None):
        for tc in m.tool_calls:
            print(f"     🔧 tool_call -> {tc['name']}({tc['args']})")

---
## ⌨️ Part 5: The String Shorthand

Passing a bare string as `messages` is accepted shorthand — LangChain wraps it into a single
`HumanMessage` for you. Handy for quick experiments, but prefer the explicit list form in real
code, where you need system messages and multi-turn history anyway.

In [ ]:
# ============================================================================
# STRING SHORTHAND: {"messages": "..."} instead of a list of dicts
# ============================================================================
# Note the deliberate typo ("New Yourk") — the model still resolves the city,
# a nice illustration that tool ARGS are model-generated, not string-parsed.
response1 = agent.invoke({"messages": "What is the weather in New Yourk"})

print("🤖 Final answer:", response1["messages"][-1].content)

In [ ]:
# ============================================================================
# MESSAGE TRAIL: Same four-message structure as before
# ============================================================================
response1["messages"]

---
## 📝 Summary

In this notebook, we learned:

### 1. Version Matters
- **LangChain 1.x only**: `langchain.agents.create_agent` does not exist on 0.x — assert the
  version at the top rather than debugging an `ImportError` later

### 2. `create_agent` Replaces Everything
- **One entry point**: supersedes `AgentExecutor`, `initialize_agent`, and
  `langgraph.prebuilt.create_react_agent`
- **Explicit arguments**: `model`, `tools`, `system_prompt` — no hidden prompt scaffolding
- **Plain functions are tools**: the docstring becomes the description, type hints become the
  argument schema

### 3. Invocation and Output
- **Input shape**: `{"messages": [{"role": "user", "content": "..."}]}`, or just a string
- **Output**: the full state dict; `response["messages"][-1].content` is the answer

### 4. Reading the Message Trail
- **Four messages per tool call**: Human → AI (tool call) → Tool → AI (answer)
- **The "empty" AI message**: it holds the tool call in `.tool_calls`, not in `.content` —
  print both when debugging

### Next Steps
- **`2-modelintegration.ipynb`** — swap between OpenAI, Google Gemini, and GROQ with
  `init_chat_model`, plus streaming and batching